# Qwen3-14B-Base-LoRA ? submit Pool v7

Inference notebook for the completed Modal two-stage Qwen3-14B run. It uses the trained plain head, centred twin, and Sinkhorn postprocessing.

Training completed on Modal A100-80GB, 2026-11-07. CV / Kaggle score pending.

## Attach on Kaggle

1. Competition data: **Name That LLM**
2. Base model: **Qwen3-14B-Base** (same as training)
3. Adapter dataset: [zuhairsan/qwen3-14b-poolv7-20261107](https://www.kaggle.com/datasets/zuhairsan/qwen3-14b-poolv7-20261107)

Checkpoint path:

`/kaggle/input/datasets/zuhairsan/qwen3-14b-poolv7-20261107`

That folder must contain `adapter/`, `norms.safetensors`, and `readouts.npz`. `tokenizer/` is optional.

## GPU settings

Select **T4 x2**, internet off. Restart the Kaggle session after the previous OOM, then run all cells. The FP16 backbone is split across both GPUs with batch size 1. LoRA updates are merged in small chunks; normalization and concatenation of pooled features run on CPU. No additional dataset is needed. Transformers and Accelerate must be available in the Kaggle environment.


## 0. Imports & paths


In [ ]:
import gc
import glob
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()

# Recipe constants (must match Qwen3-14B-Base-Lora training)
KEEP = 40
MAX_LEN = 447
LAYERS = [5, 10, 15, 20, 25, 30, 35, 40]
BS = 1  # Keep activation memory small on 2 x T4
TWIN = dict(thresh=0.85, min_size=2, max_size=18)

# Blend: plain head weight + centred-twin weight (twin only on topic clusters)
W_PLAIN = 1.0
W_TWIN = 0.35


def find_data_csv(name: str) -> Path:
    candidates = [
        Path('/kaggle/input/competitions/name-that-llm') / name,
        Path('/kaggle/input/name-that-llm') / name,
        Path('data') / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in ('/kaggle/input', '.'):
        if Path(root).exists():
            hits = sorted(Path(root).rglob(name))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)


CKPT_DIR = Path("/kaggle/input/datasets/zuhairsan/qwen3-14b-poolv7-20261107")


def find_job_final() -> Path:
    """pool_v7 adapter from zuhairsan/qwen3-14b-poolv7-20261107."""
    p = CKPT_DIR
    missing = [
        name
        for name, ok in (
            ("adapter/", (p / "adapter").is_dir()),
            ("readouts.npz", (p / "readouts.npz").is_file()),
            ("norms.safetensors", (p / "norms.safetensors").is_file()),
        )
        if not ok
    ]
    if missing:
        raise FileNotFoundError(
            f"{p} is missing {missing}. Attach zuhairsan/qwen3-14b-poolv7-20261107."
        )
    return p


def find_qwen3_14b_base() -> str:
    search_roots = []
    if Path('/kaggle/input').is_dir():
        search_roots.append('/kaggle/input/**/config.json')
    search_roots.append('**/config.json')
    for pat in search_roots:
        for cfg in sorted(glob.glob(pat, recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if (
                c.get('model_type') == 'qwen3'
                and c.get('hidden_size') == 5120
                and c.get('num_hidden_layers') == 40
            ):
                return os.path.dirname(cfg)
    raise FileNotFoundError('Attach Qwen3-14B-Base under /kaggle/input')


def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


CKPT = find_job_final()
BASE_DIR = find_qwen3_14b_base()
train = pd.read_csv(find_data_csv('train.csv'))
test = pd.read_csv(find_data_csv('test.csv'))
test['text'] = test['text'].fillna('').astype(str)

with np.load(CKPT / 'readouts.npz', allow_pickle=False) as z:
    LABELS = [str(x) for x in z['classes'].tolist()]
    POOL_LAYERS = [int(x) for x in z['pool_layers'].tolist()]
    PLAIN_W = z['plain_coef'].astype(np.float64)
    PLAIN_B = z['plain_intercept'].astype(np.float64)
    CENTER_W = z['center_coef'].astype(np.float64)
    CENTER_B = z['center_intercept'].astype(np.float64)

assert POOL_LAYERS == LAYERS, (POOL_LAYERS, LAYERS)
K = len(LABELS)
assert PLAIN_W.shape == (K, 5120 * len(LAYERS)), PLAIN_W.shape

print('checkpoint', CKPT)
print('base', BASE_DIR)
print('labels', LABELS)
print('train', train.shape, 'test', test.shape, flush=True)


## 1. Load base + merge LoRA + extract pooled features

Same pooling as training: mean over tokens (mask pad + first token) at layers 5/10/15/20/25/30/35/40, each through its LayerNorm, then concatenate.

Unlike the two-stage public checkpoints, we **keep** the model’s final RMSNorm (training did not replace it with `Identity`).


In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def load_trunk(base_dir: str, keep: int, dtype):
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    if torch.cuda.device_count() < 2:
        raise RuntimeError('Select Kaggle GPU T4 x2 and restart the session.')
    # The embedding table adds weight to GPU 0, so it gets fewer layers.
    device_map = {'embed_tokens': 0, 'rotary_emb': 0, 'norm': 1}
    device_map.update({f'layers.{i}': 0 if i < 19 else 1 for i in range(keep)})
    model = AutoModel.from_pretrained(
        base_dir, config=mc, dtype=dtype, device_map=device_map,
        low_cpu_mem_usage=True, attn_implementation='sdpa',
    ).eval()
    print('Model split across GPUs:', model.hf_device_map, flush=True)
    # do NOT set model.norm = Identity — host LoRA24 training kept the default norm
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir: Path):
    acfg = json.load(open(adapter_dir / 'adapter_config.json'))
    scale = acfg['lora_alpha'] / acfg['r']
    sd = load_file(str(adapter_dir / 'adapter_model.safetensors'))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if '.lora_A.' not in k:
            continue
        name = k.split('.lora_A.')[0]
        # peft keys look like base_model.model.layers.0.... ; keep from "layers."
        lin = modules[name[name.index('layers.') :]]
        Bm = sd[k.replace('.lora_A.', '.lora_B.')]
        # Merge on CPU in row chunks to avoid allocating a large FP32 delta
        # on a nearly full T4. Each chunk goes to its own layer's GPU.
        A = A.float().cpu()
        Bm = Bm.float().cpu()
        for start in range(0, Bm.shape[0], 256):
            delta = (Bm[start:start + 256] @ A) * scale
            lin.weight[start:start + 256].add_(
                delta.to(device=lin.weight.device, dtype=lin.weight.dtype))
            del delta


@torch.no_grad()
def extract_features(texts, bs: int = BS) -> np.ndarray:
    dtype = torch.float16  # Match training; T4 does not support native BF16
    tok_dir = CKPT / 'tokenizer'
    tok = AutoTokenizer.from_pretrained(str(tok_dir if tok_dir.is_dir() else BASE_DIR))
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = load_trunk(BASE_DIR, KEEP, dtype)
    merge_lora(model, CKPT / 'adapter')

    norms = nn.ModuleList([nn.LayerNorm(5120) for _ in LAYERS]).cpu().float()
    nsd = load_file(str(CKPT / 'norms.safetensors'))
    # training saved keys as norms.0.weight ; ModuleList expects 0.weight
    norms.load_state_dict({k.split('norms.', 1)[-1]: v for k, v in nsd.items()})

    ids = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )['input_ids']
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind='stable')
    Z = np.zeros((len(ids), 5120 * len(LAYERS)), np.float32)

    for s in range(0, len(order), bs):
        idx = order[s : s + bs]
        enc = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to('cuda:0')
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        parts = []
        for layer, nm in zip(LAYERS, norms):
            h = hs[layer].float()
            if not torch.isfinite(h).all():
                raise RuntimeError(f'Nonfinite features at layer {layer}; cannot score this batch.')
            m = enc['attention_mask'].to(h.device).unsqueeze(-1).float()
            m[:, 0] = 0
            pooled = ((h * m).sum(1) / m.sum(1).clamp_min(1)).cpu()
            parts.append(nm(pooled))
            del h, m, pooled
        Z[idx] = torch.cat(parts, dim=-1).numpy()
        del hs, enc, parts
        if s % 100 == 0:
            print(f'Features: {min(s + bs, len(order))}/{len(order)}', flush=True)

    del model, norms
    torch.cuda.empty_cache()
    gc.collect()
    return Z


Z = extract_features(test.text.tolist(), bs=BS)
print('Z', Z.shape, f'({time.time() - T0:.0f}s)', flush=True)
assert np.isfinite(Z).all()


## 2. Centred twin (topic clusters)

Cluster test passages by content TF‑IDF. Inside clusters of size 2–18, subtract the cluster mean from features and score with the **center** head (trained on same-prompt triplet centering).


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer

Xt = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(test.text.tolist())
if len(test) >= 2:
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(
        linkage(squareform(D, checks=False), method='average'),
        TWIN['thresh'],
        criterion='distance',
    )
    del D
else:
    groups = np.ones(len(test), dtype=int)

size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN['min_size']) & (size <= TWIN['max_size'])
print(
    f'topic clusters: {int((size == 1).sum())} singletons, '
    f'{int(twin_rows.sum())} passages in clusters of 2-18',
    flush=True,
)

plain_logits = Z @ PLAIN_W.T + PLAIN_B
LOGIT = W_PLAIN * np.log(np.clip(softmax(plain_logits), 1e-9, 1))

if twin_rows.any():
    Zc = Z.copy()
    for g in np.unique(groups[twin_rows]):
        Zc[groups == g] -= Zc[groups == g].mean(0)
    center_logits = Zc[twin_rows] @ CENTER_W.T + CENTER_B
    LOGIT[twin_rows] += W_TWIN * np.log(np.clip(softmax(center_logits), 1e-9, 1))

print(f'heads applied ({time.time() - T0:.0f}s)', flush=True)


## 3. Sinkhorn balance + write `submission.parquet`

Rescale so each of the 18 classes gets ~equal mass on the full test set (skipped for the 3-row interactive `test.csv`).


In [ ]:
P = softmax(LOGIT)
if len(test) >= 50:
    target = np.full(K, len(P) / K)
    for _ in range(100):
        P *= target / np.clip(P.sum(0), 1e-12, None)
        P /= np.clip(P.sum(1, keepdims=True), 1e-12, None)

sub = pd.DataFrame(
    {
        'id': test['id'].astype(str),
        'model': np.array(LABELS)[P.argmax(1)],
    }
)
assert sub['id'].is_unique
assert set(sub['model']).issubset(set(LABELS))
sub.to_parquet('submission.parquet', index=False)

print(sub['model'].value_counts().sort_index().to_string())
print(f'done in {time.time() - T0:.0f}s → submission.parquet ({len(sub)} rows)')
